# This notebook assigns data weights.

When data are distributed very unevenly, for example because of station clusters or earthquake clusters, TomoATT built-in regularization alone may not fully compensate for the uneven data distribution.

We can therefore assign external data weights to adjust the contribution from different regions. The core rule is:

The denser the data, the lower the weight of each individual datum.

For differential data, the weight is the average of the two associated absolute-arrival weights.

In [ ]:
# Import TomoATT data-processing utilities
import sys
sys.path.append('../utils')
import functions_for_data as ffd

# 1. Read the Arrival-Time Data File

In [ ]:
# Read data
fname = "output_data/step4_src_rec_Cs.dat"
[ev_info_obs, st_info_obs] = ffd.read_src_rec_file(fname)

# Data plot with weights (optional); set fname = None to skip saving the figure.
ffd.fig_ev_st_distribution_wt(ev_info_obs, st_info_obs, fname = None)

# 2. Assign Weights to Earthquakes

Two weighting schemes are provided:

### (a) Box Weighting (recommended)

Divide the study region into many equally spaced subregions. If a subregion contains N earthquakes, each earthquake in that subregion is assigned a weight of 1 / sqrt(N).

The logic is that regions with more data still have a larger total weight, N * 1 / sqrt(N) = sqrt(N), but each individual datum has a smaller weight, 1 / sqrt(N).

In [ ]:
# Use box weighting with subregion size 0.4 degrees x 0.4 degrees x 5 km.
dlon = 0.4; dlat = 0.4; ddep = 5
ev_box = ffd.box_weighting_ev(ev_info_obs,dlon,dlat,ddep)

# Data plot with weights (optional); set fname = None to skip saving the figure.
ffd.fig_ev_st_distribution_wt(ev_box, st_info_obs, fname = "./figs/step5_ev_box_wt.png")

### (b) Geographic Weighting (optional)

Reference:

Ruan, Y., Lei, W., Modrak, R., Örsvuran, R., Bozdağ, E., & Tromp, J. (2019). Balancing unevenly distributed data in seismic tomography: a global adjoint tomography example. Geophysical Journal International, 219(2), 1225-1236.

In this scheme, the weight of each earthquake is positively correlated with its distance to all other earthquakes.

The logic is that events near the edge of the study region or in sparse areas have larger average distances to other events and therefore receive larger weights.

Note 1: This weighting scheme is more suitable for global tomography. In local tomography, under similar data density, it tends to emphasize events near the edge of the study region because those events are farther from other events on average.

Note 2: This scheme is more aggressive than box weighting.

In [ ]:
# See the English README for details.
dlon = 0.2; dlat = 0.2; ddep = 5
ev_geo = ffd.geographical_weighting_ev_rough(ev_info_obs,dlon,dlat,ddep)

# Plot the weight distribution
ffd.fig_ev_st_distribution_wt(ev_geo, st_info_obs, fname = "./figs/step5_ev_geo_wt.png")

# 3. Assign Weights to Stations

The same two weighting schemes can be applied to stations and are not repeated here.

In [ ]:
# Apply box weighting to stations
dlon = 0.2; dlat = 0.2; 
(ev_box,st_box) = ffd.box_weighting_st(ev_box,st_info_obs,dlon,dlat)

# Plot the weight distribution
ffd.fig_ev_st_distribution_wt(ev_box, st_box, fname = "./figs/step5_ev_st_box_wt.png")

In [ ]:
# Apply geographic weighting to stations
# Because the number of stations is usually small, pairwise station distances can be computed directly.
(ev_geo,st_geo) = ffd.geographical_weighting_st(ev_geo,st_info_obs)

# Plot the weight distribution
ffd.fig_ev_st_distribution_wt(ev_geo, st_geo, fname = "./figs/step5_ev_st_geo_wt.png")

# 4. Output Files

In [ ]:
# Write data to the target directory
import os

# Specify the data directory
out_path = "output_data"
os.makedirs(out_path,exist_ok=True)

# Save as a TomoATT-format data file
out_fname = "%s/step5_src_rec_weight.dat"%(out_path)
ffd.write_src_rec_file(out_fname,ev_info_obs,st_info_obs)

# Earthquake list for plotting
out_fname_ev = "%s/step5_ev_list_weight.dat"%(out_path)
ffd.write_src_list_file(out_fname_ev,ev_info_obs)

# Station list for plotting
out_fname_st = "%s/step5_st_list_weight.dat"%(out_path)
ffd.write_rec_list_file(out_fname_st,ev_info_obs,st_info_obs)